[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/student/Lab3_LangGraph_Workflows_Student.ipynb)

# Lab 3: LangGraph — Stateful Workflows
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> Build a clinical triage system: a patient presents with symptoms, the system classifies severity (low/medium/high), routes to appropriate handling (self-care advice, schedule appointment, or emergency alert), and produces a structured output.

### Objective
- Build a `StateGraph` with typed state
- Create nodes that read and update state
- Add conditional routing based on state
- Visualize the workflow graph

---
## Setup

In [ ]:
!pip install -q langchain langchain-openai langgraph grandalf

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete.")

---
## Cell 1 — Define the State Schema

In LangGraph, the **state** is a typed dictionary that flows through the graph. Each node can read from and write to the state.

Think of it as the patient's chart that gets passed from station to station in the triage process.

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import add_messages

# TODO: Define the TriageState TypedDict
# It should have:
#   - patient_symptoms: str (the patient's presenting symptoms)
#   - severity: str (will be set to "low", "medium", or "high")
#   - recommendation: str (the triage recommendation)
#   - messages: Annotated[list, add_messages] (LLM conversation history)
#
# Hint: class TriageState(TypedDict):
#           patient_symptoms: str
#           ...

class TriageState(TypedDict):
    # YOUR CODE HERE
    pass

print("State schema defined.")
print("Fields:", list(TriageState.__annotations__.keys()))

---
## Cell 2 — Define the Nodes

Each node is a function that:
1. Receives the current state
2. Does some work (may call an LLM)
3. Returns a dict with the state keys it wants to update

Our triage workflow has 5 nodes:
- **intake_node**: Records the patient's symptoms
- **classify_node**: Uses LLM to classify severity
- **low_severity_node**: Provides self-care advice
- **medium_severity_node**: Schedules an appointment
- **high_severity_node**: Triggers emergency alert

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# Node 1: Intake — records symptoms into state
def intake_node(state: TriageState) -> dict:
    """Process patient intake — record symptoms."""
    print("[INTAKE] Processing patient symptoms...")
    return {
        "messages": [
            SystemMessage(content="You are a clinical triage nurse at KHCC."),
            HumanMessage(content=f"Patient presents with: {state['patient_symptoms']}")
        ]
    }

# Node 2: Classify — uses LLM to determine severity
# TODO: Complete the classify_node function
# It should:
# 1. Call the LLM with structured output to classify severity
# 2. Return a dict updating the "severity" key in state
#
# Hint: Use llm.with_structured_output() with a Pydantic model

from pydantic import BaseModel, Field

class SeverityClassification(BaseModel):
    """Classification of patient symptom severity."""
    severity: Literal["low", "medium", "high"] = Field(
        description="Severity level: low=self-care, medium=appointment needed, high=emergency"
    )
    reasoning: str = Field(description="Brief clinical reasoning for the classification")

def classify_node(state: TriageState) -> dict:
    """Classify symptom severity using LLM."""
    print("[CLASSIFY] Analyzing symptom severity...")

    # TODO: Create a structured LLM and classify the symptoms
    # Hint:
    #   classifier = llm.with_structured_output(SeverityClassification)
    #   result = classifier.invoke(f"Classify the severity... {state['patient_symptoms']}")
    #   return {"severity": result.severity}

    # YOUR CODE HERE
    pass

# Node 3: Low severity — self-care advice
def low_severity_node(state: TriageState) -> dict:
    """Handle low severity — provide self-care advice."""
    print("[LOW] Generating self-care advice...")
    response = llm.invoke(
        f"As a triage nurse, provide brief self-care advice for: {state['patient_symptoms']}. "
        f"Keep it to 2-3 sentences. Mention when to seek further care."
    )
    return {"recommendation": f"SELF-CARE: {response.content}"}

# Node 4: Medium severity — schedule appointment
# TODO: Complete the medium_severity_node
# It should call the LLM to generate an appointment recommendation
# and return {"recommendation": f"APPOINTMENT: {response.content}"}

def medium_severity_node(state: TriageState) -> dict:
    """Handle medium severity — schedule appointment."""
    print("[MEDIUM] Scheduling appointment...")
    # YOUR CODE HERE
    pass

# Node 5: High severity — emergency alert
# TODO: Complete the high_severity_node
# It should call the LLM to generate emergency instructions
# and return {"recommendation": f"EMERGENCY: {response.content}"}

def high_severity_node(state: TriageState) -> dict:
    """Handle high severity — emergency alert."""
    print("[HIGH] EMERGENCY ALERT!")
    # YOUR CODE HERE
    pass

print("All nodes defined.")

---
## Cell 3 — Define Conditional Edges

After classification, the graph needs to route to the correct handler based on severity. This is done with a **conditional edge**.

In [ ]:
# TODO: Define the routing function
# It should read state["severity"] and return the name of the next node
#   "low" -> "low_handler"
#   "medium" -> "medium_handler"
#   "high" -> "high_handler"
#
# Hint: def route_by_severity(state: TriageState) -> str:
#           severity = state["severity"]
#           if severity == "low":
#               return "low_handler"
#           ...

def route_by_severity(state: TriageState) -> str:
    """Route to the appropriate handler based on severity."""
    # YOUR CODE HERE
    pass

print("Routing function defined.")

---
## Cell 4 — Build and Compile the Graph

Now we assemble the nodes and edges into a `StateGraph`.

In [ ]:
from langgraph.graph import StateGraph, START, END

# TODO: Build the graph step by step
# 1. Create the StateGraph with TriageState
# 2. Add nodes: "intake", "classify", "low_handler", "medium_handler", "high_handler"
# 3. Add edges:
#    START -> "intake"
#    "intake" -> "classify"
#    "classify" -> conditional edge (route_by_severity)
#    "low_handler" -> END
#    "medium_handler" -> END
#    "high_handler" -> END
# 4. Compile the graph
#
# Hint:
#   graph = StateGraph(TriageState)
#   graph.add_node("intake", intake_node)
#   graph.add_node("classify", classify_node)
#   ...
#   graph.add_edge(START, "intake")
#   graph.add_edge("intake", "classify")
#   graph.add_conditional_edges("classify", route_by_severity)
#   graph.add_edge("low_handler", END)
#   ...
#   triage_app = graph.compile()

# YOUR CODE HERE

print("Graph compiled successfully!")

---
## Cell 5 — Run with Different Patient Scenarios

Let's test our triage system with patients of different severity levels.

In [ ]:
# Test scenarios
scenarios = [
    {
        "name": "Scenario 1: Mild symptoms",
        "symptoms": "Mild headache for 2 days, no fever, no vision changes. Patient is otherwise healthy."
    },
    {
        "name": "Scenario 2: Moderate symptoms",
        "symptoms": "Persistent cough for 2 weeks, low-grade fever (37.8C), unintentional weight loss of 3kg. Former smoker."
    },
    {
        "name": "Scenario 3: Severe symptoms",
        "symptoms": "Sudden severe chest pain, difficulty breathing, sweating, pain radiating to left arm. History of hypertension."
    }
]

# TODO: Run each scenario through the triage graph
# Hint: for scenario in scenarios:
#           result = triage_app.invoke({
#               "patient_symptoms": scenario["symptoms"],
#               "severity": "",
#               "recommendation": "",
#               "messages": []
#           })

for scenario in scenarios:
    print("\n" + "=" * 60)
    print(f"  {scenario['name']}")
    print("=" * 60)
    print(f"Symptoms: {scenario['symptoms'][:80]}...")
    print()

    # YOUR CODE HERE: invoke triage_app
    result = # YOUR CODE HERE

    print(f"\nSeverity: {result['severity'].upper()}")
    print(f"Recommendation: {result['recommendation'][:300]}")

---
## Cell 6 — Visualize the Graph

LangGraph can generate a visual representation of your workflow.

In [ ]:
# Visualize the graph as ASCII
print("Graph structure:")
print()
triage_app.get_graph().print_ascii()

# You can also get a Mermaid diagram for rendering
print("\n\nMermaid diagram (paste into mermaid.live):")
print(triage_app.get_graph().draw_mermaid())

---
## Stretch Challenge

1. Add a **follow-up node** after each handler that generates a structured JSON summary:
   ```python
   class TriageSummary(BaseModel):
       patient_symptoms: str
       severity: str
       recommendation: str
       follow_up_days: int
       red_flags_to_watch: list[str]
   ```
2. Add **memory** using `MemorySaver` so the triage system can remember previous patient encounters:
   ```python
   from langgraph.checkpoint.memory import MemorySaver
   memory = MemorySaver()
   triage_app = graph.compile(checkpointer=memory)
   ```
3. Run the same patient through twice and see if the system references the prior visit.

### KHCC Connection
This triage workflow pattern is directly applicable at KHCC:
- Emergency department triage for oncology patients
- Phone triage for patients calling with symptoms between treatments
- Chemotherapy side-effect severity assessment
- Routing patients to the right clinic (medical oncology, radiation, surgical, palliative)
- The graph can be extended with more nodes for lab ordering, specialist referral, etc.